# CASCADES v10 — Elastic Riemannian Ecosystem

Full training pipeline with 5 theoretical advancements:
1. **GQA-Aware Metric Preconditioning** — resolves 8B scaling paradox
2. **Ambient Trace Dedup** — fixes hidden geometric flaw in sleep
3. **Tikhonov Soft-EAR** — smooth gradient reassignment
4. **Principal Tangent Expansion** — noise-free rank revival
5. **Subspace-Contrastive Decoding** — adapter-level CFG

**Training Data:** Real benchmarks (CommonsenseQA → ARC-Challenge → GSM8K)

**Runtime:** ~10 min on A100

## 1. Setup

In [ ]:
!git clone https://github.com/Bender1011001/CASCADES--continual-PEFT-for-Local-LLMs.git
%cd CASCADES--continual-PEFT-for-Local-LLMs
!pip install -q -r requirements.txt

In [ ]:
#@title 1.5 · Upload Abliterated Qwen3 Model { run: "auto" }
# ---------------------------------------------------------------
# Upload abliteratedqwen3.zip (from Obliteratus) and extract it
# to abliterated/ so train_qwen35.py can find it at MODEL_PATH.
# ---------------------------------------------------------------
import os, zipfile, shutil
from pathlib import Path

MODEL_ZIP = "abliteratedqwen3.zip"
MODEL_DIR = "abliterated"

if os.path.isdir(MODEL_DIR) and any(Path(MODEL_DIR).iterdir()):
    print(f"✓ {MODEL_DIR}/ already present — skipping upload")
else:
    # Option A: load from Google Drive (faster on A100)
    drive_path = f"/content/drive/MyDrive/{MODEL_ZIP}"
    if os.path.exists(drive_path):
        print(f"Found {MODEL_ZIP} on Google Drive — copying...")
        shutil.copy(drive_path, MODEL_ZIP)
    else:
        print(f"Select {MODEL_ZIP} to upload:")
        from google.colab import files
        uploaded = files.upload()
        if MODEL_ZIP not in uploaded:
            raise FileNotFoundError(
                f"Expected '{MODEL_ZIP}', got: {list(uploaded.keys())}\n"
                "Rename your zip to abliteratedqwen3.zip and re-run."
            )

    print(f"Extracting to {MODEL_DIR}/...")
    os.makedirs(MODEL_DIR, exist_ok=True)
    with zipfile.ZipFile(MODEL_ZIP, "r") as zf:
        members = zf.namelist()
        # If zip has one top-level folder, flatten it into MODEL_DIR
        top_dirs = {m.split("/")[0] for m in members if "/" in m}
        if len(top_dirs) == 1:
            top = top_dirs.pop()
            for member in members:
                rel = os.path.relpath(member, top)
                if rel == ".":
                    continue
                dest = os.path.join(MODEL_DIR, rel)
                if member.endswith("/"):
                    os.makedirs(dest, exist_ok=True)
                else:
                    os.makedirs(os.path.dirname(dest), exist_ok=True)
                    with zf.open(member) as src, open(dest, "wb") as dst:
                        dst.write(src.read())
        else:
            zf.extractall(MODEL_DIR)

    print(f"✓ Extracted to {MODEL_DIR}/")
    print(f"  Contents: {sorted(f.name for f in Path(MODEL_DIR).iterdir())}")


## 2. Verify GPU

In [ ]:
!nvidia-smi

import torch
print(f"\nPyTorch: {torch.__version__}")
print(f"CUDA: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    props = torch.cuda.get_device_properties(0)
    vram = props.total_memory / 1024**3
    print(f"VRAM: {vram:.1f} GB")
else:
    print("No GPU! Go to Runtime > Change runtime type > A100 GPU")

## 3. Verify v10 Library

In [ ]:
import cascades
print(f"CASCADES version: {cascades.__version__}")
assert cascades.__version__ == '2.0.0', f"Expected v2.0.0, got {cascades.__version__}"

# Verify v10 features are importable
from cascades.math_ops import gqa_precondition_gradient, soft_ear
from cascades.config import AblationConfig

cfg = AblationConfig()
print(f"GQA ratio: {cfg.gqa_ratio}")
print(f"Soft-EAR gamma: {cfg.ear_gamma}")
print(f"CFG lambda: {cfg.cfg_lambda}")
print(f"Principal expansion: {cfg.enable_principal_expansion}")
print("\nAll v10 features verified.")

## 4. Download Real Benchmark Data

Downloads 3 real benchmark datasets from HuggingFace:
- **Task 0:** GSM8K (grade-school math, natural CoT)
- **Task 1:** ARC-Challenge (science reasoning)
- **Task 2:** CommonsenseQA (commonsense reasoning)

**300 examples per task** (doubled from 150): more diversity reduces task-boundary forgetting and gives the Stiefel manifold a richer null-space to project against.

In [ ]:
!python download_real_data.py --samples 300

OUTPUT_PREFIX = "cascades_hx"

# train_qwen35.py loads MODEL_PATH="abliterated" (extracted in step 1.5)
# and uses inject_hybrid_cascades() with correct Qwen3 layer names + NF4 quant.
!python train_qwen35.py \
    --epochs 2 \
    --output_prefix {OUTPUT_PREFIX}


In [ ]:
OUTPUT_PREFIX = "cascades_v10_colab"

!python train.py \
    --epochs 2 \
    --dmole_threshold 0.30 \
    --output_prefix {OUTPUT_PREFIX}

## 6. Results

In [ ]:
import pandas as pd
import numpy as np

results_csv = f"{OUTPUT_PREFIX}_results.csv"
df = pd.read_csv(results_csv, index_col=0)

print("Accuracy Matrix (Proxy ACC %):")
print((df * 100).round(2).to_string())

# Backward Transfer
num_tasks = len(df)
bwt_vals = [df.iloc[-1, i] - df.iloc[i, i] for i in range(num_tasks - 1)]
bwt = np.mean(bwt_vals) * 100
avg_acc = df.iloc[-1].mean() * 100

print(f"\nAverage Accuracy: {avg_acc:.2f}%")
print(f"Backward Transfer: {bwt:+.2f}%")
print(f"\nVerdict: {'Zero forgetting confirmed' if bwt >= 0 else 'Some forgetting detected'}")

## 7. EM Evaluation with CFG Decoding

Runs generative evaluation using v10 subspace-contrastive decoding.

In [ ]:
!python evaluate.py --weights {OUTPUT_PREFIX}_weights.pt --fast --max_samples 10 --max_new_tokens 512

## 8. Download Weights

Download the trained v10 adapter weights to use locally.

In [ ]:
from google.colab import files
files.download(f"{OUTPUT_PREFIX}_weights.pt")
files.download(f"{OUTPUT_PREFIX}_results.csv")